In [ ]:
!pip install -U "transformers>=4.44.0" \
              "peft>=0.11.0" \
              "bitsandbytes>=0.43.0" \
              datasets accelerate einops mamba-ssm sentencepiece\
              "trl==0.9.6"

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    Mamba2ForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    AutoModelForCausalLM
)
from trl import SFTTrainer,DataCollatorForCompletionOnlyLM
from peft import LoraConfig
import json
import pandas as pd
import re
import os

In [ ]:


# --- CONFIGURAZIONE PATH ---
root = "/content/drive/MyDrive/NLP/TAP-IFTTT"

SYNTH_DATA_PATH  = os.path.join(root, "data/dataset/applets/applets_synt_train.jsonl")
BASE_OUTPUT_DIR  = os.path.join(root, "data/results_instruct")
OUTPUT_FILE=os.path.join(root, "data/dataset/applets/applets_synt_train_processed.jsonl")
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
applet = pd.read_json(SYNTH_DATA_PATH, lines=True,orient='records')

print("Numero totale applet:", len(applet))

input_col_name = "istruct_ft"

output_col_name = "filter_code"

SEPARATOR = "<|INST|>"   # separatore chiaro, senza newline ambigui

applet["text"] = (
    applet[input_col_name].astype(str)
    + SEPARATOR +
    applet[output_col_name].astype(str)
)

print("Esempio 'text':")
print(applet["text"].iloc[0])

# 6) Salva JSONL con solo la colonna 'text'
records = applet[["text"]].to_dict(orient="records")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")


In [2]:
# --- CONFIGURAZIONE MODELLO ---
model_id = "mistralai/Mamba-Codestral-7B-v0.1"
arch = "mamba"
model_id="codellama/CodeLlama-7b-Instruct-hf"
arch="llama"
model_id="Qwen/Qwen2.5-Coder-7B-Instruct"
arch='qwen'
model_id="google/codegemma-7b-it"
arch='gemma'
model_id="deepseek-ai/deepseek-coder-6.7b-instruct"
arch='deepseek'

DATA_FILE = OUTPUT_FILE

# 4-bit config (QLoRA-style)
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    compute_dtype = torch.bfloat16
else:
    compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    from_slow=True,
    legacy=False,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # per Mamba2 va bene; warning TRL è solo un avviso

# Modello 4-bit
#Mamba2ForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False  # importante per il training

def get_lora_target_modules(architecture: str, available_layers=None):
   
    arch = architecture.lower()

    if arch in ["llama", "deepseek","qwen"]:
        return ["q_proj", "k_proj", "v_proj", "o_proj"]
    if "gemma" in arch:
        return ["query_proj", "key_proj", "value_proj", "output_proj"]
    if "mamba" in arch or "codestral" in arch:
        # Core sicuri per tutti i Mamba2
        targets = ["in_proj", "out_proj"]
        if available_layers and "x_proj" in available_layers:
            targets.insert(0, "x_proj")
        return targets
    return ["q_proj", "k_proj", "v_proj", "o_proj"]

# LoRA config (senza out_proj per Mamba2)
lora_config = LoraConfig(
    r=32,                      # più piccolo per risparmiare memoria
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=get_lora_target_modules(),
)

# Dataset
dataset = load_dataset(
    "json",
    data_files=DATA_FILE,
    split="train",
)

SEP_STR = SEPARATOR

def find_sep_ids_from_example(example_text: str):
    # sanity check
    if SEP_STR not in example_text:
        raise ValueError("SEP_STR non trovato nell'esempio, controlla il dataset")

    # posizione in caratteri
    sep_char_pos = example_text.index(SEP_STR)

    # testo prima e prima+sep
    before = example_text[:sep_char_pos]
    before_and_sep = example_text[:sep_char_pos + len(SEP_STR)]

    # tokenizza senza special tokens
    ids_before = tokenizer.encode(before, add_special_tokens=False)
    ids_before_and_sep = tokenizer.encode(before_and_sep, add_special_tokens=False)

    # la differenza sono esattamente i token di SEP nel contesto reale
    sep_ids = ids_before_and_sep[len(ids_before):]
    return sep_ids

# prendiamo un esempio qualsiasi (puoi usare anche più di uno, ma 1 basta)
ex_text = dataset[0]["text"]
sep_ids = find_sep_ids_from_example(ex_text)

# collator: passiamo GLI ID, non la stringa
collator = DataCollatorForCompletionOnlyLM(
    tokenizer=tokenizer,
    response_template=sep_ids, 
)

fp16 = compute_dtype == torch.float16
bf16 = compute_dtype == torch.bfloat16

training_args = TrainingArguments(
    output_dir="./mamba-mqlora-results",
    num_train_epochs=5,
    per_device_train_batch_size=1,   # 1 per ridurre il picco di memoria
    gradient_accumulation_steps=4,   # batch effettivo ~8
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    fp16=fp16,
    bf16=bf16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config,
    data_collator=collator,
    tokenizer=tokenizer,        # <-- torniamo alla vecchia API (ok anche se deprecata)
    dataset_text_field="text",  # <-- sì, deprecato ma ancora supportato
    max_seq_length=1024,         # <-- accorciato per evitare OOM
    packing=False,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/256 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:408: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:413: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 2}.


Step,Training Loss
10,1.359100
20,0.517300
30,0.405400
40,0.371800
50,0.320000
60,0.269200
70,0.178900
80,0.197300
90,0.183000
100,0.117000


TrainOutput(global_step=320, training_loss=0.16988041079603136, metrics={'train_runtime': 7818.7364, 'train_samples_per_second': 0.164, 'train_steps_per_second': 0.041, 'total_flos': 1.309361779740672e+16, 'train_loss': 0.16988041079603136, 'epoch': 5.0})

In [3]:
normalized = re.sub(r"[.-]", "_", model_id.split("/")[-1])

adapter_dir = f"/content/drive/MyDrive/NLP/TAP-IFTTT/models/COLAB_{normalized}"
os.makedirs(adapter_dir, exist_ok=True)
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("Adapter LoRA salvato in:", adapter_dir)


Adapter LoRA salvato in: /content/drive/MyDrive/NLP/TAP-IFTTT/models/COLAB_mamba-codestral-7b-applets-qlora


In [4]:
from google.colab import drive
drive.flush_and_unmount()